# Lecture: Variational Autoencoder (VAE) on Fashion-MNIST

This notebook applies the same VAE introduced for MNIST (notebook **C2-3**) to **Fashion-MNIST**: 60,000 training images of size 28x28 in ten clothing classes (T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot).

The model architecture (`VAE` from `VAE.py`) and the training procedure are **identical** to C2-3 — only the dataset changes. Fashion-MNIST is more challenging than MNIST: higher intra-class variance and less sharp inter-class boundaries. This makes the latent-space structure more interesting to inspect and the generative quality of the VAE easier to judge visually.

If you have not seen the VAE theory yet, work through **C2-3** first.

Run the following cell only if you are working with Google Colab to copy the required `VAE.py` file into the working directory. If you work locally, just ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C2-Autoencoders/VAE.py ./

### Data Preparation and ELBO Loss

In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from VAE import VAE

device     = "cuda" if torch.cuda.is_available() else "cpu"
LATENT_DIM = 2
epochs     = 20

transform = transforms.Compose([transforms.ToTensor()])

FASHION_CLASSES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Ankle boot"
]

def elbo_loss(x, x_hat, mu, log_var):
    n_pixels = x.shape[1] * x.shape[2] * x.shape[3]
    recon = F.mse_loss(x_hat, x, reduction="mean")
    kl    = (-0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum(dim=1)).mean() / n_pixels
    return recon + kl, recon, kl

fashion_train = FashionMNIST(root="./data", train=True,  download=True, transform=transform)
fashion_test  = FashionMNIST(root="./data", train=False, download=True, transform=transform)

fashion_train_loader = DataLoader(fashion_train, batch_size=256, shuffle=True,  num_workers=0, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test,  batch_size=256, shuffle=False, num_workers=0)

print(f"Device: {device}")
print(f"Training samples: {len(fashion_train)}, Test samples: {len(fashion_test)}")

### Training

Same architecture and hyperparameters as before. Because Fashion-MNIST is more complex, the reconstruction loss settles at a higher value — the 2D bottleneck is a tight constraint for ten visually diverse classes.

In [ ]:
fashion_model     = VAE(latent_dim=LATENT_DIM).to(device)
fashion_optimizer = optim.Adam(fashion_model.parameters(), lr=1e-3)

for epoch in range(epochs):
    fashion_model.train()
    total_loss = total_recon = total_kl = 0

    for x, _ in fashion_train_loader:
        x = x.to(device, non_blocking=True)

        fashion_optimizer.zero_grad()
        x_hat, mu, log_var = fashion_model(x)
        loss, recon, kl = elbo_loss(x, x_hat, mu, log_var)
        loss.backward()
        fashion_optimizer.step()

        total_loss  += loss.item()
        total_recon += recon.item()
        total_kl    += kl.item()

    n = len(fashion_train_loader)
    print(f"Epoch {epoch+1:2d}  "
          f"loss={total_loss/n:.5f}  "
          f"recon={total_recon/n:.5f}  "
          f"kl={total_kl/n:.5f}")

In [ ]:
fashion_model.save_model(path="models/vae_fashion_mnist.pth")

If you do not want to train, load the pre-trained model (latent_dim=2, 20 epochs, full Fashion-MNIST training set).

In [ ]:
fashion_model = VAE(latent_dim=LATENT_DIM).to(device)
#fashion_model.load_model(path="models/vae_fashion_mnist.pth", device=device) # for running locally
fashion_model.load_model(path="AIBIP/C2-Autoencoders/models/vae_fashion_mnist.pth", device=device) # for running in colab

### Reconstruction Quality

Reconstructions are noticeably blurrier than on MNIST — a known limitation of VAEs with MSE loss on complex textures. The model captures the overall shape and class identity but loses fine detail.

In [ ]:
fashion_model.eval()

x_batch, labels_batch = next(iter(fashion_test_loader))
x_batch = x_batch[:10].to(device)

with torch.no_grad():
    x_hat, _, _ = fashion_model(x_batch)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x_batch[i].squeeze().cpu(), cmap="gray")
    axes[0, i].axis("off")
    axes[0, i].set_title(FASHION_CLASSES[labels_batch[i].item()], fontsize=6)
    axes[1, i].imshow(x_hat[i].squeeze().cpu(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original",       fontsize=10)
axes[1, 0].set_ylabel("Reconstruction", fontsize=10)
plt.tight_layout()
plt.show()

### Latent Space

The 2D latent space shows less clean separation than MNIST — visually similar classes (Shirt vs. T-shirt, Sneaker vs. Ankle boot) overlap strongly. The KL regularisation still enforces a roughly standard-normal distribution.

In [ ]:
fashion_model.eval()

all_mu     = []
all_labels = []

with torch.no_grad():
    for x, y in fashion_test_loader:
        mu, _ = fashion_model.encoder(x.to(device))
        all_mu.append(mu.cpu().numpy())
        all_labels.append(y.numpy())

all_mu     = np.concatenate(all_mu,     axis=0)
all_labels = np.concatenate(all_labels, axis=0)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(all_mu[:, 0], all_mu[:, 1],
                     c=all_labels, cmap="tab10", s=2, alpha=0.6)
cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
cbar.ax.set_yticklabels(FASHION_CLASSES, fontsize=7)
ax.set_xlabel("z[0]")
ax.set_ylabel("z[1]")
ax.set_title("2D Latent Space (posterior means) — Fashion-MNIST Test Set")
plt.tight_layout()
plt.show()

### Sampling from the Prior

Samples drawn from $p_\theta(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I})$ and decoded by the Fashion-MNIST VAE. The generated items are recognisable as clothing, though blurrier than real examples — typical for VAEs with pixel-wise MSE loss.

In [ ]:
fashion_model.eval()

samples = fashion_model.sample(16, device=device).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle("Samples from $p_\\theta(z) = \\mathcal{N}(0, I)$ — Fashion-MNIST VAE", y=1.02)
plt.tight_layout()
plt.show()

---
## Exercise: The Effect of the Latent Dimension on Fashion-MNIST

In notebook **C2-3** you saw on MNIST that raising the latent dimension from 2
to 16 improved reconstruction quality. **Does the same hold on the harder
Fashion-MNIST data — and what happens to prior sampling?**

Your task:

1. **Train** a second Fashion-MNIST VAE with `latent_dim = 16` (the cell below is
   ready — you only change `EX_LATENT_DIM`). On a Colab GPU this takes a few
   minutes. If you cannot train, a pre-trained checkpoint is provided as a
   fallback.
2. **Compare reconstructions** of the `latent_dim = 2` model (`fashion_model`,
   trained/loaded above) with your new `latent_dim = 16` model.
3. **Compare prior samples** from both models.

Answer these questions:

- Does `latent_dim = 16` improve reconstruction as clearly as it did on MNIST,
  or less? Why might Fashion-MNIST behave differently?
- Do the **prior samples** get better, worse, or just different when you raise
  the latent dimension? (Recall the fidelity-vs-prior trade-off — a higher
  latent dimension does not automatically give better samples.)

In [ ]:
# --- Train a latent_dim=16 Fashion-MNIST VAE (or load the fallback below) ---
EX_LATENT_DIM = 16   # <-- the only change vs. the dim=2 model trained above

ex_model     = VAE(latent_dim=EX_LATENT_DIM).to(device)
ex_optimizer = optim.Adam(ex_model.parameters(), lr=1e-3)

for epoch in range(epochs):
    ex_model.train()
    total_loss = total_recon = total_kl = 0
    for x, _ in fashion_train_loader:
        x = x.to(device, non_blocking=True)
        ex_optimizer.zero_grad()
        x_hat, mu, log_var = ex_model(x)
        loss, recon, kl = elbo_loss(x, x_hat, mu, log_var)
        loss.backward()
        ex_optimizer.step()
        total_loss  += loss.item()
        total_recon += recon.item()
        total_kl    += kl.item()
    n = len(fashion_train_loader)
    print(f"Epoch {epoch+1:2d}  loss={total_loss/n:.5f}  "
          f"recon={total_recon/n:.5f}  kl={total_kl/n:.5f}")

ex_model.save_model(path="models/vae_fashion_mnist_latent16.pth")

If you do not want to train, load the pre-trained `latent_dim=16` Fashion-MNIST model (20 epochs) instead.

In [ ]:
EX_LATENT_DIM = 16
ex_model = VAE(latent_dim=EX_LATENT_DIM).to(device)
ex_model.load_model(path="models/vae_fashion_mnist_latent16.pth", device=device) # for running locally
#ex_model.load_model(path="AIBIP/C2-Autoencoders/models/vae_fashion_mnist_latent16.pth", device=device) # for running in colab
ex_model.eval()

In [ ]:
# Compare reconstructions: latent_dim=2 vs latent_dim=16
fashion_model.eval(); ex_model.eval()

x_batch, labels_batch = next(iter(fashion_test_loader))
x_batch = x_batch[:10].to(device)

with torch.no_grad():
    x2,  _, _ = fashion_model(x_batch)
    x16, _, _ = ex_model(x_batch)

fig, axes = plt.subplots(3, 10, figsize=(15, 4.5))
for i in range(10):
    axes[0, i].imshow(x_batch[i].squeeze().cpu(), cmap="gray"); axes[0, i].axis("off")
    axes[0, i].set_title(FASHION_CLASSES[labels_batch[i].item()], fontsize=6)
    axes[1, i].imshow(x2[i].squeeze().cpu(),      cmap="gray"); axes[1, i].axis("off")
    axes[2, i].imshow(x16[i].squeeze().cpu(),     cmap="gray"); axes[2, i].axis("off")
axes[0, 0].set_ylabel("Original",        fontsize=10)
axes[1, 0].set_ylabel("latent_dim = 2",  fontsize=10)
axes[2, 0].set_ylabel("latent_dim = 16", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Compare prior samples: latent_dim=2 vs latent_dim=16
with torch.no_grad():
    s2  = fashion_model.sample(8, device=device).cpu()
    s16 = ex_model.sample(8, device=device).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(s2[i].squeeze(),  cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(s16[i].squeeze(), cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("dim = 2",  fontsize=10)
axes[1, 0].set_ylabel("dim = 16", fontsize=10)
plt.suptitle("Prior samples  z ~ N(0, I)  — Fashion-MNIST")
plt.tight_layout()
plt.show()

---
## Try It Yourself — Experiment with Fashion-MNIST

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. Most-confused classes.** Which two clothing classes does the VAE confuse
most often? Inspect the latent scatter plot — do they overlap there? Why these
two in particular?

**B. Tune $\beta$ yourself.** In `elbo_loss`, change the KL weight to `beta = 0.5`
and then `beta = 4` (multiply the `kl` term by `beta`). Re-train for just 5
epochs each — that is enough to see the trend. How do **reconstruction** *and*
**prior samples** change? Relate your observation to the $\beta$-VAE slide.

**C. Harder data, smaller gain.** You measured that raising `latent_dim` from 2
to 16 helps *less* on Fashion-MNIST than on MNIST. Give one concrete reason why
the harder dataset behaves differently.

**D. Sampling trade-off.** Did the `latent_dim=16` prior samples get *better*,
*worse*, or just *different* than `latent_dim=2`? Explain using the
fidelity-vs-prior trade-off.